In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re 

import altair as alt

alt.__version__

# plots

import matplotlib.pyplot as plt
import numpy as np

import vl_convert


In [2]:
# Notebook is inside I2R/notebooks
PROJECT = Path("..").resolve()

DATA_RAW = PROJECT / "data" / "raw"
DATA_PROCESSED = PROJECT / "data" / "processed"
OUTPUTS = PROJECT / "outputs"

print("Project root:", PROJECT)
print("Raw data path:", DATA_RAW)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

print("Paths ready:", DATA_RAW)


Project root: C:\Users\Michelle\I2R
Raw data path: C:\Users\Michelle\I2R\data\raw
Paths ready: C:\Users\Michelle\I2R\data\raw


In [3]:
# Load csv train

DATA_TRAIN = PROJECT / "data" / "train"

csv_train = DATA_TRAIN / "Warehouse_and_Retail_Sales.csv"

df_train = pd.read_csv(csv_train)

df_train

,YEAR,MONTH,SUPPLIER,ITEM CODE,ITEM DESCRIPTION,ITEM TYPE,RETAIL SALES,RETAIL TRANSFERS,WAREHOUSE SALES
0,2020,1,REPUBLIC NATIONAL DISTRIBUTING CO,100009,BOOTLEG RED - 750ML,WINE,0.00,0.0,2.00
1,2020,1,PWSWN INC,100024,MOMENT DE PLAISIR - 750ML,WINE,0.00,1.0,4.00
2,2020,1,RELIABLE CHURCHILL LLLP,1001,S SMITH ORGANIC PEAR CIDER - 18.7OZ,BEER,0.00,0.0,1.00
3,2020,1,LANTERNA DISTRIBUTORS INC,100145,SCHLINK HAUS KABINETT - 750ML,WINE,0.00,0.0,1.00
4,2020,1,DIONYSOS IMPORTS INC,100293,SANTORINI GAVALA WHITE - 750ML,WINE,0.82,0.0,0.00
...,...,...,...,...,...,...,...,...,...
307640,2020,9,LEGENDS LTD,99753,DUTCHESS DE BOURGOGNE NR - 750ML,BEER,0.00,0.0,5.00
307641,2020,9,ANHEUSER BUSCH INC,9997,HOEGAARDEN 4/6NR - 12OZ,BEER,66.12,37.0,240.75
307642,2020,9,COASTAL BREWING COMPANY LLC,99970,DOMINION OAK BARREL STOUT 4/6 NR - 12OZ,BEER,2.25,0.0,0.00
307643,2020,9,BOSTON BEER CORPORATION,99990,SAM ADAMS SUMMER VARIETY 12PK NR,BEER,20.50,0.0,0.00


In [4]:
# date

# Create date column (first day of month)
df_train["date"] = pd.to_datetime(
    dict(year=df_train["YEAR"].astype("Int64"), month=df_train["MONTH"].astype("Int64"), day=1),
    errors="coerce"
)

# pie chart


In [5]:
# define chart id + meta helper

import json
from datetime import datetime
from uuid import uuid4

def new_chart_id(prefix="pie"):
    # short but unique
    return f"{prefix}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid4().hex[:8]}"

def save_metadata(meta: dict, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2, default=str)

In [6]:
# pie data samples 

#choose random month with pos total sales
# group by item type (or later supplier)
#keep top-k + other
# reject degenerate cases


def sample_pie_data(
    df: pd.DataFrame,
    rng: np.random.Generator,
    date_col="date",
    category_col="ITEM TYPE",
    value_col="RETAIL SALES",
    k_min=3,
    k_max=8,
    min_total=1.0,
):
    # 1) choose a valid month (positive totals)
    monthly_totals = df.groupby(date_col)[value_col].sum()
    valid_months = monthly_totals[monthly_totals > min_total].index.to_numpy()
    if len(valid_months) == 0:
        raise ValueError("No valid months with positive totals found.")

    month = pd.to_datetime(rng.choice(valid_months))

    # 2) aggregate within month by category
    s = (
        df.loc[df[date_col] == month]
          .groupby(category_col)[value_col]
          .sum()
          .sort_values(ascending=False)
    )

    # remove zeros / negatives (pie needs non-negative)
    s = s[s > 0]

    # Need enough categories
    if len(s) < k_min:
        return None  # signal "resample"

    # 3) choose top_k
    top_k = int(rng.integers(k_min, min(k_max, len(s)) + 1))
    top = s.head(top_k).copy()
    other_sum = s.iloc[top_k:].sum()

    if other_sum > 0:
        top.loc["Other"] = other_sum

    # final sanity
    total = float(top.sum())
    if total <= min_total or len(top) < k_min:
        return None

    # return a plotting table + context
    plot_df = top.reset_index()
    plot_df.columns = [category_col, value_col]

    context = {
        "month": month,
        "top_k": top_k,
        "total": total,
        "n_categories_raw": int(len(s)),
    }
    return plot_df, context

In [7]:
# generate altail charts and export to svg

# plot function

def make_pie_chart_altair(plot_df, category_col, value_col, title=None):
    chart = (
        alt.Chart(plot_df, title=title)
        .mark_arc()
        .encode(
            theta=alt.Theta(field=value_col, type="quantitative"),
            color=alt.Color(field=category_col, type="nominal"),
            tooltip=[category_col, value_col]
        )
    )
    return chart

In [8]:
# save svg

def save_altair_svg(chart, svg_path):
    svg_path = Path(svg_path)
    svg_path.parent.mkdir(parents=True, exist_ok=True)
    chart.save(str(svg_path), format="svg")

In [9]:
def generate_pie(
    df: pd.DataFrame,
    out_root: Path,
    dataset_source: str,
    rng_seed: int = 42,
    date_col="date",
    category_col="ITEM TYPE",
    value_col="RETAIL SALES",
    max_tries: int = 50,
):
    rng = np.random.default_rng(rng_seed)
    chart_id = new_chart_id("pie")

    for attempt in range(1, max_tries + 1):
        sampled = sample_pie_data(
            df=df,
            rng=rng,
            date_col=date_col,
            category_col=category_col,
            value_col=value_col,
        )
        if sampled is None:
            continue

        plot_df, context = sampled

        # --- Save the table used for plotting
        table_path = out_root / "tables" / f"{chart_id}.csv"
        plot_df.to_csv(table_path, index=False)

        # --- Build & save chart
        month_str = pd.to_datetime(context["month"]).strftime("%Y-%m")
        title = f"{value_col} share by {category_col} ({month_str})"
        chart = make_pie_chart_altair(plot_df, category_col, value_col, title=title)

        svg_path = out_root / "images" / f"{chart_id}.svg"
        save_altair_svg(chart, svg_path)

        # --- Metadata
        meta = {
            "chart_id": chart_id,
            "chart_type": "pie",
            "dataset_source": dataset_source,
            "created_utc": datetime.utcnow().isoformat() + "Z",
            "columns_used": {
                "date": date_col,
                "category": category_col,
                "value": value_col,
            },
            "data_filters": {
                "month": month_str,
                "value_positive_only": True,
                "top_k_plus_other": True,
            },
            "sampling": {
                "attempt": attempt,
                "n_categories_raw": context["n_categories_raw"],
                "top_k": context["top_k"],
                "total_value": context["total"],
            },
            "outputs": {
                "svg": str(svg_path.as_posix()),
                "table_csv": str(table_path.as_posix()),
            },
            "style_parameters": {
                # keep empty for now; later you’ll sample from your priors
            },
        }

        meta_path = out_root / "meta" / f"{chart_id}.json"
        save_metadata(meta, meta_path)

        return meta

    raise RuntimeError(f"Failed to generate a valid pie chart after {max_tries} attempts.")

In [10]:
# sanity test
OUT_ROOT = PROJECT / "data" / "generated" / "pie"
dataset_source = "Warehouse and Retail Sales (your saved file / data.gov link)"

meta = generate_pie(
    df=df_train,
    out_root=OUT_ROOT,
    dataset_source=dataset_source,
    rng_seed=123
)

meta

{'chart_id': 'pie_20260224T135629_ba126927',
 'chart_type': 'pie',
 'dataset_source': 'Warehouse and Retail Sales (your saved file / data.gov link)',
 'created_utc': '2026-02-24T13:56:33.522211Z',
 'columns_used': {'date': 'date',
  'category': 'ITEM TYPE',
  'value': 'RETAIL SALES'},
 'data_filters': {'month': '2017-06',
  'value_positive_only': True,
  'top_k_plus_other': True},
 'sampling': {'attempt': 1,
  'n_categories_raw': 6,
  'top_k': 5,
  'total_value': 97357.26},
 'outputs': {'svg': 'C:/Users/Michelle/I2R/data/generated/pie/images/pie_20260224T135629_ba126927.svg',
  'table_csv': 'C:/Users/Michelle/I2R/data/generated/pie/tables/pie_20260224T135629_ba126927.csv'},
 'style_parameters': {}}

In [11]:
# make 10 pie charts 

metas = []

for i in range(10):
    meta = generate_pie(
        df=df_train,
        out_root=OUT_ROOT,
        dataset_source="Warehouse and Retail Sales",
        rng_seed=1000 + i   # different seed per chart
    )
    metas.append(meta)

print("Generated:", len(metas), "charts")
print("First chart ID:", metas[0]["chart_id"])

Generated: 10 charts
First chart ID: pie_20260224T135744_55575fad
